In [1]:
import sys

import numpy as np

sys.path.append(r"C:\Users\zschende\OneDrive - Ilmatieteen laitos\Data\smrt")

In [2]:
import numpy as np

from smrt import make_atmosphere, make_model, make_snowpack, make_soil, sensor_list
from smrt.core.globalconstants import PERMITTIVITY_OF_AIR


In [3]:
def forest_transmissivity_cohen2015(frequency, theta, stem_volume, mode="P"):
    """Calculate transmissivity for forest canopy according to Cohen et al. (2015).
    Args:
        frequency (list of floats): Frequency in Hz.
        theta (float or list of floats): Incidence angle in degrees.
        stem_volume (int, optional): Forest stem volume in m3/ha.
        mode (str, optional): Only passive mode ("P") is implemented, active mode ("A") could be added.
    Raises:
        ValueError: An error occurs when the frequencies are specified outside the valid range of the model.
    Returns:
        dict: Transmissivity for each frequency (["V", "H"]) and theta
    """

    costheta = np.cos(theta)

    # passive mode
    if mode == "P":  # pola = ["V", "H"]
        kappa_e = np.zeros((len(frequency), 2))  # effective extinction coefficient

        for id, freq in enumerate(frequency):
            if freq < 17e9:
                kappa_e[id, :] = np.array([[0, 0]])  # negligible attenuation
            elif freq > 17e9 and freq < 20e9:
                kappa_e[id, :] = np.array([[0.007, 0.010]])
            elif freq > 35e9 and freq < 39e9:
                kappa_e[id, :] = np.array([[0.011, 0.012]])
            else:
                raise ValueError("Frequency not supported")

        for id, angle in enumerate(theta):
            ke = kappa_e * np.cos(np.deg2rad(50))  # extinction coefficient (model derived at 50 degrees)

            if id == 0:
                transmissivity_oneway = np.exp((-ke * stem_volume) / costheta[id])
            else:
                transmissivity_oneway = np.concatenate(
                    (transmissivity_oneway, np.exp((-ke * stem_volume) / costheta[id])), axis=1
                )
        transmissivity = {freq: transmissivity_oneway[id] for id, freq in enumerate(frequency)}

    # active mode placeholder
    elif mode == "A":
        pass

    return transmissivity

In [ ]:
# passive microwave sensor
sensor = sensor_list.cimr(channel=["18", "37"], theta=[40, 50])

# canopy transmissivity calculation
transmissivity = forest_transmissivity_cohen2015(
    frequency=sensor.frequency, theta=sensor.theta, stem_volume=10, mode="P"
)

In [9]:
# sample dicts
tb_down = {18.7e9: [25, 40], 36.5e9: [15, 20]}
tb_up = {18.7e9: [10, 25], 36.5e9: [23, 38]}

# simple atmosphere with polarization-sensitive transmissivity
atmos = make_atmosphere(
    "simple_atmosphere",
    theta=np.rad2deg(sensor.theta),
    tb_down=tb_down,
    tb_up=tb_up,
    transmittance=transmissivity,
    npol=2,  # specify!
)

# rename to highlight we understand it to represent a canopy instead of an atmosphere
canopy = atmos

In [10]:
# interpolation demo for polarization-sensitive transmissivity
print(
    np.interp(np.cos(np.deg2rad(45)), canopy.costheta, canopy.trans[sensor.frequency[0]][0::2]),  # V-pol
    np.interp(np.cos(np.deg2rad(45)), canopy.costheta, canopy.trans[sensor.frequency[0]][1::2]),  # H-pol
)
atmos.run(frequency=sensor.frequency[0], costheta=np.cos(np.deg2rad(45)), npol=2)

0.937904839204361 0.9124961279232595


AtmosphereResult(tb_down=array([32.17254293, 32.17254293]), tb_up=array([17.17254293, 17.17254293]), transmittance=array([0.93790484, 0.91249613]))

In [11]:
# frozen soil from Meloche et al. (2021)
sub = make_soil(
    "soil_wegmuller",
    permittivity_model=complex(3.3, 0.0051),
    roughness_rms=1.65e-2,
    temperature=270,
)

# sample 1-layer snowpack
sp = make_snowpack(
    thickness=np.array([1]),
    microstructure_model="exponential",
    density=[250],
    temperature=[260],
    background_permittivity_model=PERMITTIVITY_OF_AIR,
    corr_length=[1 * 4 * (1 - 300 / 917) / 917 / 25],
    substrate=sub,
    atmosphere=canopy,  # use canopy as atmosphere
)

smrt_model = make_model("iba", "dort")

TB = smrt_model.run(sensor, sp)